In [1]:
import os
from llama_index.core import set_global_handler
from dotenv import load_dotenv, find_dotenv
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from llama_index.core.tools import QueryEngineTool, ToolMetadata, FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.llms.groq import Groq
from pymongo import MongoClient, AsyncMongoClient
from llama_index.tools.tavily_research import TavilyToolSpec
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SQLDatabase, VectorStoreIndex, Settings, StorageContext
from llama_index.core.query_engine import NLSQLTableQueryEngine

load_dotenv(find_dotenv())

llm = Groq(
    model= 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY')
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)

Settings.llm = llm



# this is optional but we set it to see the "thought" process in the logs
set_global_handler("simple")
print('# 👍Imports done.')

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 👍Imports done.


In [ ]:
# we are going to create the SQLite Sample database here
# this cell creates a local company_sales.db file and populates it with sample data so you can test the quantitative routing

import sqlite3
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer, select

# creating SQlite file and connection

engine = create_engine("sqlite:///company_sales.db")
metadataObj = MetaData()

# Defining a sample table (e.g., Quarterly Revenue)
revenueTable = Table(
    "revenue_stats",
    metadataObj,
    Column("year", Integer),
    Column("quarter", String(16), primary_key=True),
    Column("revenue_millions", Integer),
)

metadataObj.create_all(engine)

# insert sample data
with engine.begin() as connection:
    connection.execute(revenueTable.insert(), [
        {"year": 2023, "quarter": "Q1", "revenue_millions": 94836},
        {"year": 2023, "quarter": "Q2", "revenue_millions": 81797},
        {"year": 2023, "quarter": "Q3", "revenue_millions": 89498},
        {"year": 2023, "quarter": "Q4", "revenue_millions": 119575}
    ])

print("SQLite Database 'company_sales.db' created with revenue_stats table.")

SQLite Database 'company_sales.db' created with revenue_stats table.


In [3]:
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

'''# creating the primary retrieval tool
apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [4]:


# ------ ENGINE 1: the SQL Brain ------
sqlDatabase = SQLDatabase(engine, include_tables= ["revenue_stats"])
sqlQueryEngine = NLSQLTableQueryEngine(sql_database= sqlDatabase)

# ---- ENGINE 2: The Vector Brain ----
vectorQueryEngine = index.as_query_engine()


In [5]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# Wrap the engines into tools
sqlTool = QueryEngineTool.from_defaults(
    query_engine= sqlQueryEngine,
    description= "Useful for quantitative questions involving exact revenue numbers, specific years, quarters, or sums/averages."
)

vectorTool = QueryEngineTool.from_defaults(
    query_engine= vectorQueryEngine,
    description= "Useful for semantic questions about business strategy, risks, company history, and qualitative facts."
)

# initialize the router
routerEngine = RouterQueryEngine(
    selector= LLMSingleSelector.from_defaults(),
    query_engine_tools= [sqlTool, vectorTool],
)

In [6]:
# we are now going to do the evaluation Run
# Test 1: Should go to SQL
print("--- TESTING SQL ROUNTING ---")
response1 = routerEngine.query("What was the tool revenue for the year 2023?")
print(f"Result: {response1}\n")

# Test 2: Should go to Vector
print("---- TESTING VECTOR ROUTING ----")
response2 = routerEngine.query("What are the primary risk factors discussed in the report?")
print(f"Result: {response2}")

--- TESTING SQL ROUNTING ---
Result: To find the total tool revenue for the year 2023, we need to sum up the revenues from the SQL response. 

The SQL response is: [(94836,), (81797,), (89498,), (119575,)]

First, extract the revenue values: 94836, 81797, 89498, 119575.

Then, sum these values: 94836 + 81797 = 176633, 176633 + 89498 = 266131, 266131 + 119575 = 385706.

Therefore, the total tool revenue for the year 2023 is 385706 million.

---- TESTING VECTOR ROUTING ----
Result: The primary risk factors are not explicitly stated in the provided information, but it is mentioned that the Company's business, reputation, results of operations, financial condition, and stock price can be affected by a number of factors. These factors are described as being able to materially and adversely affect the Company when they materialize. However, the specific risk factors themselves are not listed in the given text.
